In [6]:
import matplotlib.pyplot as plt
import numpy as np
import glob
import os
from google.colab import files
from datetime import datetime
import zipfile
import shutil

# Загрузка файлов
print("Загрузите файлы:")
uploaded = files.upload()

# Определяем типы файлов (можно добавлять любые)
file_types = {
    'j_e_t=*.txt': 'j_e_t_plots',
    'j_i_t=*.txt': 'j_i_t_plots',
    'potential_t=*.txt': 'potential_t_plots',
}

# Создаём основную папку
main_folder = f'plots_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
os.makedirs(main_folder, exist_ok=True)
print(f"Папка: {main_folder}/")

# Функция загрузки и фильтрации с улучшенным удалением выбросов
def load_data(file_path):
    data = np.loadtxt(file_path)
    if len(data.shape) == 1:
        data = data.reshape(1, -1)
    x = data[:, 0]
    y = data[:, 1]

    # Способ 1: Удаляем значения, которые резко отличаются (по стандартному отклонению)
    # Вычисляем порог: median + 5 * std (более робастный метод)
    median_y = np.median(y)
    std_y = np.std(y)
    threshold = 5 * std_y

    # Способ 2: Простой порог (можно настроить под свои данные)
    # threshold = 1e10  # Для j_e_t файлов

    # Создаём маску: значения не должны быть выбросами
    mask = np.abs(y - median_y) < threshold

    # Дополнительно: если после фильтрации осталось мало точек, используем менее строгий порог
    if np.sum(mask) < len(y) * 0.5:
        print(f"  ⚠ {os.path.basename(file_path)}: слишком много выбросов, использую порог 1e10")
        mask = np.abs(y) < 1e10

    return x[mask], y[mask]

# Альтернативная функция с удалением последней точки (если она сильно отличается)
def load_data_alternative(file_path):
    data = np.loadtxt(file_path)
    if len(data.shape) == 1:
        data = data.reshape(1, -1)
    x = data[:, 0]
    y = data[:, 1]

    # Удаляем последнюю точку, если она является выбросом
    if len(y) > 1 and np.abs(y[-1]) > 10 * np.mean(np.abs(y[:-1])):
        print(f"  ⚠ {os.path.basename(file_path)}: удалена последняя точка-выброс ({y[-1]:.2e})")
        x = x[:-1]
        y = y[:-1]

    # Дополнительная фильтрация по порогу
    mask = np.abs(y) < 1e10
    return x[mask], y[mask]

# Выберите одну из функций (по умолчанию используем улучшенную)
# load_data = load_data  # улучшенная
# load_data = load_data_alternative  # альтернативная

# Обработка каждого типа файлов
for pattern, subfolder in file_types.items():
    files = glob.glob(pattern)
    if not files:
        continue

    # Создаём подпапку
    subfolder_path = os.path.join(main_folder, subfolder)
    os.makedirs(subfolder_path, exist_ok=True)
    print(f"\n{subfolder}/ ({len(files)} файлов)")

    for file_path in files:
        x, y = load_data(file_path)

        if len(x) == 0:
            print(f"  ✗ {os.path.basename(file_path)}: нет данных после фильтрации")
            continue

        plt.figure(figsize=(14, 7))
        plt.plot(x, y, 'b-', linewidth=1.5, marker='.', markersize=3)
        plt.xlabel('Первый столбец')
        plt.ylabel('Второй столбец')
        plt.title(f'График: {os.path.basename(file_path)}')
        plt.grid(True, alpha=0.3)
        plt.axhline(y=0, color='r', alpha=0.3)

        # Добавляем информацию об удалённых выбросах
        original_len = len(np.loadtxt(file_path))
        if len(y) < original_len:
            plt.text(0.02, 0.98, f'Удалено выбросов: {original_len - len(y)}',
                     transform=plt.gca().transAxes, verticalalignment='top',
                     fontsize=9, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

        plt.tight_layout()

        save_path = os.path.join(subfolder_path,
                                os.path.basename(file_path).replace('.txt', '.png'))
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  ✓ {os.path.basename(save_path)} (точек: {len(y)})")
        plt.close()

# Удаление исходных файлов
print("\nУдаление исходных .txt файлов...")
for f in glob.glob('*.txt'):
    try:
        os.remove(f)
        print(f"✓ Удалён: {f}")
    except Exception as e:
        print(f"✗ Ошибка: {e}")

# Создание ZIP архива
print("\nСоздание ZIP архива...")
zip_name = f'{main_folder}.zip'
with zipfile.ZipFile(zip_name, 'w') as zipf:
    for root, dirs, files_in_dir in os.walk(main_folder):
        for file in files_in_dir:
            zipf.write(os.path.join(root, file),
                      os.path.relpath(os.path.join(root, file), os.path.dirname(main_folder)))
print(f"✓ ZIP архив: {zip_name}")

# Удаление папки
shutil.rmtree(main_folder)
print(f"✓ Папка удалена: {main_folder}")

print(f"\n✅ Готово! {zip_name} создан")

Загрузите файлы:


Saving j_e_t=0.000000001446.txt to j_e_t=0.000000001446.txt
Saving j_e_t=0.000000001808.txt to j_e_t=0.000000001808.txt
Saving j_e_t=0.000000002169.txt to j_e_t=0.000000002169.txt
Saving j_e_t=0.000000002531.txt to j_e_t=0.000000002531.txt
Saving j_e_t=0.000000002892.txt to j_e_t=0.000000002892.txt
Saving j_e_t=0.000000003254.txt to j_e_t=0.000000003254.txt
Saving j_e_t=0.000000003615.txt to j_e_t=0.000000003615.txt
Saving j_e_t=0.000000003977.txt to j_e_t=0.000000003977.txt
Saving j_e_t=0.000000004338.txt to j_e_t=0.000000004338.txt
Saving j_e_t=0.000000004700.txt to j_e_t=0.000000004700.txt
Saving j_e_t=0.000000010122.txt to j_e_t=0.000000010122.txt
Saving j_e_t=0.000000010303.txt to j_e_t=0.000000010303.txt
Saving j_e_t=0.000000010484.txt to j_e_t=0.000000010484.txt
Saving j_e_t=0.000000010665.txt to j_e_t=0.000000010665.txt
Saving j_e_t=0.000000010845.txt to j_e_t=0.000000010845.txt
Saving j_e_t=0.000000011026.txt to j_e_t=0.000000011026.txt
Saving j_e_t=0.000000011207.txt to j_e_t